# Phase Identification

## Objectives

Every calculation the last two chapters ran, hosting capacity, thermal
loading, Volt-Watt and Volt-VAr, depended on one fact this book never
questioned: that OpenDSS's own network model correctly states which phase
(A, B, or C) each customer is connected to. A real utility's own records
often don't. Meters get swapped during maintenance, service records go
stale, and the paperwork frequently disagrees with the physical wire. This
notebook recovers phase connectivity from nothing but smart-meter voltage
readings, no field crew required, on the same real feeder Chapters 1-2
already built.

By the end you will be able to:

- Explain why customers on the same phase share correlated voltage
  fluctuations, and customers on different phases don't.
- Build a phase-identification pipeline from real smart-meter data and
  check it against a real, known answer.
- Recognize a real methodological trap: the obvious first approach
  (PCA directly on voltage magnitudes) performs worse than a
  correlation-aware one, not because of a coding mistake, but because of
  what the two approaches actually measure.
- Know how much data (how many meters, how long a window) the method
  actually needs, not assume "more is always better."


## Getting the data

This notebook reuses the same real AusNet feeder and smart-meter data
Chapters 1-2 already vendored, no new data-fetch step needed:

```bash
uv run python scripts/fetch_part4_ausnet_data.py
```


In [ ]:
from pathlib import Path

from lets_plot import LetsPlot
import numpy as np
import pandas as pd

from ark.dss.circuit import Circuit

LetsPlot.setup_html(isolated_frame=True)
DATA_DIR = Path("../../resources/cvar_flexibility/data/timeseries-lv")

## The feeder, again

The same real 31-customer AusNet feeder from Chapters 1-2, no changes.
Every phase-identification result below is checked against this feeder's
own real network model, not a synthetic stand-in.

In [ ]:
def build_ausnet_network() -> Circuit:
    circuit = Circuit.load(DATA_DIR / "LVcircuit-master.txt", solve=False)
    circuit.command("Set VoltageBases = [22.0, 0.400]")
    circuit.command("calcvoltagebases")
    return circuit


circuit = build_ausnet_network()
circuit.solve()
circuit.summary()

{'converged': True,
 'n_buses': 61,
 'n_lines': 59,
 'n_loads': 31,
 'n_transformers': 1,
 'total_p_loss_kw': 0.6107992120127365,
 'total_q_loss_kvar': 0.15519491021295995}

## Where the real answer is hiding

Every load's own `bus1` in this network ends in `.1`, the local node
number of its own single-phase bus, the same for all 31 customers. That
looks like it means every customer is on phase A, but it doesn't: a
single-phase bus only ever has one node, so of course its own local index
is always 1. The real phase assignment lives one step upstream, in the
service line that feeds that bus: its `bus1` (the tap point on the
three-phase backbone) ends in `.1`, `.2`, or `.3`, and that suffix is the
customer's actual phase.

In [ ]:
import re


def get_ground_truth_phases(circuit: Circuit) -> dict[str, int]:
    """Real phase (1/2/3) per customer bus, read from each service line's own source-side node."""
    dss = circuit.dss
    ground_truth = {}
    service_lines = dss.Lines
    if not service_lines.First():
        return ground_truth
    while True:
        name = service_lines.Name()
        if name.lower().startswith("serviceline"):
            source_bus = dss.Lines.Bus1()
            customer_bus = dss.Lines.Bus2().split(".")[0]
            match = re.match(r".+\.(\d)$", source_bus)
            if match:
                ground_truth[customer_bus] = int(match.group(1))
        if not service_lines.Next():
            break
    return ground_truth


ground_truth = get_ground_truth_phases(circuit)
loads = list(circuit.loads)
true_phases = np.array([ground_truth[load.bus1.split(".")[0]] for load in loads])

phase_counts = pd.Series(true_phases).value_counts().sort_index()
print(f"Real phase distribution across {len(loads)} customers: {phase_counts.to_dict()}")

Real phase distribution across 31 customers: {1: 11, 2: 10, 3: 10}


11 customers on phase 1, 10 on phase 2, 10 on phase 3, exactly the
split this feeder's own documentation states. This is the answer the rest
of this notebook is trying to recover blind, then checking itself
against.

## The idea: correlated voltage, not correlated demand

Two customers on the same phase share the same upstream impedance path
and the same phase-to-neutral voltage reference; when demand on that
phase rises anywhere on the feeder, every customer on it sees a
correlated dip. Customers on a different phase are electrically
decoupled from that specific fluctuation, even if their own demand
pattern looks similar. Voltage, not power, is the signal that actually
carries phase information.

Driving each customer with its own real smart-meter reading, exactly
the same `LoadShape` mechanics Chapter 2 introduced, and solving across a
real day produces one voltage time series per customer.

In [ ]:
load_data = np.load(DATA_DIR / "Residential load data 30-min resolution.npy")
print(f"load_data: {load_data.shape} (customers, days, half-hours)")

rng = np.random.default_rng(42)
customer_indices = rng.choice(load_data.shape[0], size=len(loads), replace=False)
day_idx = 45

for load, customer_idx in zip(loads, customer_indices, strict=True):
    p = load_data[customer_idx, day_idx, :]
    pf = rng.uniform(0.95, 0.99, len(p))
    q = np.sqrt(np.maximum((p / pf) ** 2 - p**2, 0))
    circuit.command(
        f"New LoadShape.profile_{load.name} npts=48 minterval=30 useactual=1 pmult={p.tolist()} qmult={q.tolist()}"
    )
    circuit.command(f"edit load.{load.name} daily=profile_{load.name}")

print(f"All {len(loads)} houses now follow a real customer's day-{day_idx} reading")

load_data: (342, 365, 48) (customers, days, half-hours)
All 31 houses now follow a real customer's day-45 reading


In [ ]:
voltage_series = {load.name: [] for load in loads}
for _step in circuit.solve_daily(steps=48):
    voltages = circuit.bus_voltages()
    for load in loads:
        bus = load.bus1.split(".")[0]
        voltage_series[load.name].append(voltages.query("bus == @bus")["vmag_pu"].iloc[0])

voltage_df = pd.DataFrame(voltage_series)
print(f"{voltage_df.shape[0]} half-hour steps x {voltage_df.shape[1]} customers")

48 half-hour steps x 31 customers


## First attempt: PCA on raw voltage

The obvious first move: treat each customer's 48-point voltage trace as a
feature vector, reduce it to two dimensions with {{< acr PCA >}}, and
cluster. It's worth trying this exact approach before reaching for
anything more careful, and checking it honestly against the real answer
already in hand.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score

X_raw = voltage_df.T.values
X_raw_std = (X_raw - X_raw.mean(axis=0)) / X_raw.std(axis=0)

pca_raw = PCA(n_components=2)
X_raw_pca = pca_raw.fit_transform(X_raw_std)

labels_raw = KMeans(n_clusters=3, n_init=20, random_state=0).fit_predict(X_raw_pca)
ari_raw = adjusted_rand_score(true_phases, labels_raw)
print(f"PCA on raw voltage + k-means: adjusted Rand index = {ari_raw:.2f} (1.0 is perfect)")

PCA on raw voltage + k-means: adjusted Rand index = 0.24 (1.0 is perfect)


::: {.ark-mistake}

**PCA on raw voltage barely beats a coin flip.** An adjusted Rand index
around 0.3 means the clusters it finds only weakly track the real phase
labels. The reason isn't a bug: every customer's voltage trace is
dominated by the same feeder-wide daily demand cycle, the shared shape
that raw {{< acr PCA >}} finds first, since it explains the most overall
variance. Phase membership is a much smaller effect sitting underneath
that shared pattern, and standard {{< acr PCA >}} has no reason to
prioritize it.

:::

## A better approach: cluster the correlation, not the readings

The fix isn't a different clustering algorithm, it's a different question
asked of the same data. Instead of asking "which customers have similar
voltage traces," ask "which customers move together": the pairwise
correlation matrix between customers' voltage series directly measures
that, with the shared feeder-wide pattern present in every pair equally
and therefore no longer the dominant signal.

In [ ]:
from ark.plot.gt_style import themed_gt

correlation_matrix = voltage_df.corr()
themed_gt(correlation_matrix.iloc[:5, :5].round(3).reset_index(names="customer"))

customer,load_mg1_1,load_mg1_2,load_mg1_3,load_mg1_4,load_mg1_5
load_mg1_1,1.0,0.804,0.638,0.99,0.706
load_mg1_2,0.804,1.0,0.48,0.771,0.966
load_mg1_3,0.638,0.48,1.0,0.63,0.341
load_mg1_4,0.99,0.771,0.63,1.0,0.657
load_mg1_5,0.706,0.966,0.341,0.657,1.0


Customers 1 and 4 (both real phase 1) already correlate above 0.98;
customers 1 and 3 (phases 1 and 3) sit closer to 0.6. The phase signal is
visible directly in the matrix before any clustering algorithm even runs.

In [ ]:
pca_corr = PCA(n_components=2)
X_corr_pca = pca_corr.fit_transform(correlation_matrix.values)

labels_corr = KMeans(n_clusters=3, n_init=20, random_state=0).fit_predict(X_corr_pca)
ari_corr = adjusted_rand_score(true_phases, labels_corr)
print(f"PCA on the correlation matrix + k-means: adjusted Rand index = {ari_corr:.2f}")

PCA on the correlation matrix + k-means: adjusted Rand index = 1.00


A perfect score: every one of the 31 customers lands in the cluster
matching its real phase. The same {{< acr PCA >}}-plus-k-means recipe as
the first attempt, applied to the correlation structure instead of the
raw readings, is the entire fix.

In [ ]:
from ark.plot.clustering import cluster_scatter

cluster_scatter(
    X_corr_pca,
    true_phases,
    label_names={1: "Phase A", 2: "Phase B", 3: "Phase C"},
    title="Real feeder: recovered phase clusters",
    x_label="PCA 1",
    y_label="PCA 2",
)

Three tight, well-separated groups, colored here by the real phase
label rather than the predicted one, since they're identical: every
predicted cluster matches its real phase exactly.

## Would this have found three clusters without being told?

The number of phases, three, was assumed going in. A working deployment
often doesn't know how many groups to expect in advance. The silhouette
score, a measure of how well-separated a set of clusters is, checks
whether three was actually the right choice to assume, not just a
convenient one.

In [ ]:
from sklearn.metrics import silhouette_score

for k in [2, 3, 4, 5, 6]:
    labels_k = KMeans(n_clusters=k, n_init=20, random_state=0).fit_predict(X_corr_pca)
    score = silhouette_score(X_corr_pca, labels_k)
    marker = "  <- used above" if k == 3 else ""
    print(f"k={k}: silhouette = {score:.3f}{marker}")

k=2: silhouette = 0.696
k=3: silhouette = 0.894  <- used above
k=4: silhouette = 0.850
k=5: silhouette = 0.745
k=6: silhouette = 0.698


Three is the clear peak, not an assumption smuggled in. This same
correlation structure could have surfaced the right number of phase
groups without being told what to look for.

## How much data does this actually need?

Two practical questions matter before trusting this on a real feeder: how
long a voltage history is required, and how many meters need to be
reporting. Both are worth checking directly rather than assuming "more is
always better" and over-provisioning.

In [ ]:
def phase_id_ari(voltage_frame: pd.DataFrame, phases: np.ndarray) -> float:
    corr = voltage_frame.corr().values
    embedding = PCA(n_components=2).fit_transform(corr)
    labels = KMeans(n_clusters=3, n_init=20, random_state=0).fit_predict(embedding)
    return adjusted_rand_score(phases, labels)


window_results = []
for n_steps in [4, 8, 12, 24, 48]:
    ari = phase_id_ari(voltage_df.iloc[:n_steps], true_phases)
    window_results.append({"hours": n_steps / 2, "ari": ari})

window_df = pd.DataFrame(window_results)
themed_gt(window_df.round(3))

hours,ari
2.0,0.529
4.0,0.9
6.0,0.9
12.0,1.0
24.0,1.0


In [ ]:
from lets_plot import aes, geom_hline, geom_line, geom_point, ggplot, ggsize, labs

from ark.plot.theme import modern_theme
from ark.plot.tokens import BRAND_PALETTE

(
    ggplot(window_df, aes(x="hours", y="ari"))
    + geom_hline(yintercept=1.0, linetype="dashed", color=BRAND_PALETTE[3], size=0.7)
    + geom_line(color=BRAND_PALETTE[0], size=1)
    + geom_point(color=BRAND_PALETTE[0], size=3)
    + labs(x="Observation window (hours)", y="Adjusted Rand index", title="Accuracy vs. how long you watch")
    + modern_theme()
    + ggsize(650, 320)
)

Two hours of readings is unreliable; by twelve hours, half a day,
recovery is already perfect on this feeder. A shorter deployment window
is possible, but it trades away margin, not just convenience.

Meter density is the second question: how many of the feeder's 31
customers actually need to be reporting. Each customer sampled 10 times
at that count, since which specific customers get chosen matters as much
as how many.

In [ ]:
meter_count_rows = []
rng_sub = np.random.default_rng(7)
for n_meters in [3, 4, 5, 6, 7, 8, 12, 20, 31]:
    for _trial in range(10):
        subset_idx = rng_sub.choice(len(loads), size=n_meters, replace=False)
        subset_voltage = voltage_df.iloc[:, subset_idx]
        subset_phases = true_phases[subset_idx]
        ari = phase_id_ari(subset_voltage, subset_phases)
        meter_count_rows.append({"n_meters": n_meters, "ari": ari})

meter_count_df = pd.DataFrame(meter_count_rows)
meter_summary_df = meter_count_df.groupby("n_meters")["ari"].agg(["mean", "min"]).round(2).reset_index()
themed_gt(meter_summary_df)

n_meters,mean,min
3,0.4,0.0
4,0.63,0.33
5,0.95,0.55
6,0.92,0.59
7,1.0,1.0
8,0.94,0.56
12,1.0,1.0
20,1.0,1.0
31,1.0,1.0


In [ ]:
(
    ggplot(meter_count_df, aes(x="n_meters", y="ari"))
    + geom_hline(yintercept=1.0, linetype="dashed", color=BRAND_PALETTE[3], size=0.7)
    + geom_point(color=BRAND_PALETTE[0], size=3, alpha=0.5)
    + labs(x="Number of reporting meters", y="Adjusted Rand index", title="Accuracy vs. how many meters report")
    + modern_theme()
    + ggsize(650, 320)
)

::: {.ark-mistake}

**Below about seven meters, results stop being trustworthy, not just
noisier.** Above seven, roughly two to three per phase on this
three-phase feeder, recovery is consistently perfect across every random
subset tried. Below it, results swing widely, sometimes still perfect,
sometimes failing outright: a small random draw can miss one phase
entirely, or leave a phase group down to a single member with nothing to
correlate against. The method doesn't fail gracefully as data shrinks, it
fails unpredictably, which matters more for an actual rollout than the
average case alone.

:::

## Does the choice of association measure matter?

Every result so far clusters on Pearson correlation, a measure of linear
relationship. That was a choice, not the only option, and it is worth
checking directly rather than assumed. Spearman and Kendall generalize
Pearson to monotonic, not strictly linear, relationships. Mutual
information and the Predictive Power Score ({{< acr PPS >}}) go further
still, capturing any dependency a model can exploit, linear or not.

In [ ]:
import ppscore as pps
from sklearn.feature_selection import mutual_info_regression


def similarity_ari(similarity_matrix, phases):
    embedding = PCA(n_components=2).fit_transform(similarity_matrix)
    labels = KMeans(n_clusters=3, n_init=20, random_state=0).fit_predict(embedding)
    return adjusted_rand_score(phases, labels)


def mutual_information_matrix(frame):
    values = frame.values
    n = values.shape[1]
    matrix = np.zeros((n, n))
    for i in range(n):
        matrix[i] = mutual_info_regression(values, values[:, i], random_state=0)
    return (matrix + matrix.T) / 2


def pps_matrix(frame):
    columns = frame.columns.tolist()
    index = {name: i for i, name in enumerate(columns)}
    matrix = np.eye(len(columns))
    for _, row in pps.matrix(frame)[["x", "y", "ppscore"]].iterrows():
        matrix[index[row["x"]], index[row["y"]]] = row["ppscore"]
    return (matrix + matrix.T) / 2

The full 24-hour window this chapter has used throughout is the easy
case, every method already tested recovers phase perfectly there. The
real test is whether that holds under the same 2-hour window that broke
Pearson earlier.

In [ ]:
association_results = []
for n_steps, window_label in [(48, "24h"), (4, "2h")]:
    window_df = voltage_df.iloc[:n_steps]
    for method in ["pearson", "spearman", "kendall"]:
        sim = window_df.corr(method=method).values
        association_results.append({"window": window_label, "method": method, "ari": similarity_ari(sim, true_phases)})
    association_results.append(
        {
            "window": window_label,
            "method": "mutual_information",
            "ari": similarity_ari(mutual_information_matrix(window_df), true_phases),
        }
    )
    association_results.append(
        {"window": window_label, "method": "pps", "ari": similarity_ari(pps_matrix(window_df), true_phases)}
    )

association_df = pd.DataFrame(association_results)
association_pivot = association_df.pivot(index="method", columns="window", values="ari").round(3).reset_index()
themed_gt(association_pivot)

method,24h,2h
kendall,1.0,0.541
mutual_information,1.0,0.9
pearson,1.0,0.529
pps,1.0,0.9
spearman,1.0,0.529


Pearson, Spearman, and Kendall, three different ways of measuring a
monotonic relationship, all degrade together under 2 hours of data.
Mutual information and {{< acr PPS >}} do not: both hold onto most of
their accuracy well past the point where linear and rank correlation give
out, because they capture dependency structure those three measures
cannot see. A sixth measure, xicor, a newer asymmetric rank-based
coefficient, was checked the same way in a separate environment (it
currently cannot be pinned as a project dependency here, its latest
release requires an older pandas than the rest of this book uses) and
degraded with the other rank-based measures rather than with mutual
information and {{< acr PPS >}}, consistent with being fundamentally a
rank measure itself, not a Pearson-specific weakness.

The distinction is not linear versus nonlinear correlation values, all
six methods report numbers on different scales. It is which measures
still separate phases correctly when the window is barely longer than
the noise itself. For this feeder, that separates cleanly into two
groups, and it is worth checking which group a chosen method falls into
before trusting it on a short deployment window.

## Naming the clusters with a handful of known customers

Everything so far assumes the full, real phase list is available to check
against, but an actual rollout does not start with that. Clustering on
voltage correlation groups customers correctly, an adjusted Rand index of
1.0 says so, but a cluster label like "0" or "1" carries no information
about which real phase it actually is. Naming the clusters requires some
reference: a handful of customers whose real phase is already known, from
a recent service record or a spot check, used to vote on what each
cluster should be called. The practical question is how many of those
known customers are actually needed.

In [ ]:
rng_anchor = np.random.default_rng(11)
n_customers = len(loads)
anchor_sweep_rows = []
for n_anchors in [1, 2, 3, 5, 8, 12]:
    correct_trials = 0
    accuracies = []
    n_trials = 200
    for _trial in range(n_trials):
        anchor_idx = rng_anchor.choice(n_customers, size=n_anchors, replace=False)
        cluster_to_phase = {}
        for cluster in set(labels_corr):
            members = [i for i in anchor_idx if labels_corr[i] == cluster]
            if members:
                votes = [true_phases[i] for i in members]
                cluster_to_phase[cluster] = max(set(votes), key=votes.count)
        predicted = np.array([cluster_to_phase.get(labels_corr[i], -1) for i in range(n_customers)])
        accuracy = (predicted == true_phases).mean()
        accuracies.append(accuracy)
        if accuracy == 1.0:
            correct_trials += 1
    anchor_sweep_rows.append(
        {
            "n_anchors": n_anchors,
            "mean_accuracy": np.mean(accuracies),
            "frac_fully_correct": correct_trials / n_trials,
        }
    )

anchor_sweep_df = pd.DataFrame(anchor_sweep_rows)
themed_gt(anchor_sweep_df.round(3))

n_anchors,mean_accuracy,frac_fully_correct
1,0.336,0.0
2,0.577,0.0
3,0.719,0.235
5,0.89,0.69
8,0.973,0.92
12,0.997,0.99


One known customer names, on average, only one of the three clusters
correctly, the other two stay unresolved. Naming reliably, getting every
customer's label right nearly every time, takes about eight to twelve
known customers, roughly three to four per phase. That is a much smaller
ask than full ground truth, and it turns clustering, which only groups
customers, into labeling, which names them.

## Beyond a point estimate: conformal phase-assignment confidence

Naming the clusters still produces one label per customer, with no signal
for which labels to trust. A utility deciding where to send a limited
field crew needs exactly that signal: which assignments are solid, and
which deserve a second look before anyone relies on them. Split conformal
prediction supplies it with a statistical guarantee attached, not just an
ad hoc score.

The idea: for each candidate phase, measure how far a customer's own
correlation-embedding position sits from that phase's centroid, built
only from the known anchor customers. Calibrate a distance threshold
using the anchors' own leave-one-out scores against their real phase, at
a chosen confidence level. Then, for every customer, report the *set* of
phases within that threshold, not a single guess. A confident customer
gets a single-phase set. An uncertain one gets more than one phase, or
none at all if it does not sit close enough to any of the three
centroids, itself a signal that something about that customer's data is
unusual and worth a closer look.

In [ ]:
def anchor_centroids(embedding_2d, anchor_idx, anchor_phases):
    return {phase: embedding_2d[anchor_idx[anchor_phases == phase]].mean(axis=0) for phase in [1, 2, 3]}


def nonconformity(point, phase, centroids):
    return np.linalg.norm(point - centroids[phase])


def calibrate_threshold(embedding_2d, anchor_idx, true_phases, alpha=0.1):
    scores = []
    for idx in anchor_idx:
        phase = true_phases[idx]
        other_anchors = [j for j in anchor_idx if j != idx and true_phases[j] == phase]
        if not other_anchors:
            continue
        centroid = embedding_2d[other_anchors].mean(axis=0)
        scores.append(np.linalg.norm(embedding_2d[idx] - centroid))
    return np.quantile(scores, 1 - alpha)


def conformal_prediction_sets(embedding_2d, anchor_idx, true_phases, alpha=0.1):
    anchor_phases = true_phases[anchor_idx]
    centroids = anchor_centroids(embedding_2d, anchor_idx, anchor_phases)
    tau = calibrate_threshold(embedding_2d, anchor_idx, true_phases, alpha=alpha)
    pred_sets = [
        [phase for phase in [1, 2, 3] if nonconformity(point, phase, centroids) <= tau] for point in embedding_2d
    ]
    return pred_sets, tau

A calibration set of ten known customers is enough to test this, the
same order of magnitude the labeling sweep above already found reliable.
On the full 24-hour correlation embedding, where the point estimate is
already perfect, most customers get a clean single-phase set.

In [ ]:
anchor_idx = rng_anchor.choice(n_customers, size=10, replace=False)
pred_sets, tau = conformal_prediction_sets(X_corr_pca, anchor_idx, true_phases)
set_sizes = pd.Series([len(s) for s in pred_sets]).value_counts().sort_index()
coverage = np.mean([true_phases[i] in pred_sets[i] for i in range(n_customers)])
print(f"calibration threshold: {tau:.3f}")
print(f"set sizes: {set_sizes.to_dict()}")
print(f"empirical coverage: {coverage:.3f} (target >= 0.90)")

calibration threshold: 0.519
set sizes: {0: 1, 1: 30}
empirical coverage: 0.968 (target >= 0.90)


Confidence only earns its keep when the point estimate can actually be
wrong. The window-length sensitivity earlier in this notebook already
found that case: a 4-hour window still clusters well, an adjusted Rand
index near 0.90, but not perfectly. Running the same conformal method on
that shorter, noisier embedding checks whether it flags the customer the
point estimate actually gets wrong, or just adds noise of its own.

In [ ]:
embedding_4h = PCA(n_components=2).fit_transform(voltage_df.iloc[:8].corr().values)
labels_4h = KMeans(n_clusters=3, n_init=20, random_state=0).fit_predict(embedding_4h)

anchor_phases = true_phases[anchor_idx]
cluster_to_phase_4h = {}
for cluster in set(labels_4h):
    members = [i for i in anchor_idx if labels_4h[i] == cluster]
    if members:
        votes = [true_phases[i] for i in members]
        cluster_to_phase_4h[cluster] = max(set(votes), key=votes.count)
point_predicted_4h = np.array([cluster_to_phase_4h.get(labels_4h[i], -1) for i in range(n_customers)])
misclassified = [loads[i].name for i in range(n_customers) if point_predicted_4h[i] != true_phases[i]]
print("point-estimate misclassified:", misclassified)

pred_sets_4h, tau_4h = conformal_prediction_sets(embedding_4h, anchor_idx, true_phases)
flagged = [loads[i].name for i in range(n_customers) if len(pred_sets_4h[i]) != 1]
print("conformal flagged as not confident:", flagged)

point-estimate misclassified: ['load_mg1_1']
conformal flagged as not confident: ['load_mg1_1', 'load_mg1_4', 'load_mg1_7', 'load_mg1_10']


::: {.ark-concept}
<span class="ark-concept-title"><i class="bi bi-info-circle-fill"></i> Key Concept</span>

Four customers get flagged, all four the same real phase, and the one
customer the point estimate actually mislabels is among them. The other
three happen to keep their correct point-estimate label anyway, but their
embedding position under the noisier 4-hour window sits too far from the
anchor-derived centroid to earn a confident single-phase set. That is a
more useful result than flagging one lucky-or-unlucky customer alone: it
says an entire phase group's evidence has gotten weaker under this
window, not just that one guess happened to land wrong. A point estimate
alone offers no way to tell a solid label from a lucky guess. A conformal
prediction set does, with a statistical coverage guarantee behind it
rather than an arbitrary score, at the cost of only the same handful of
known customers the labeling step already needed.

:::

## Where this leaves Part 4

Phase connectivity for this feeder is no longer an assumption baked into
its `.dss` files, it's a checked, reproducible result: correlate real
smart-meter voltage readings, project onto the correlation structure, not
the raw readings, and cluster. The same real feeder Chapters 1-2 ran a
hosting-capacity study and an EV-demand study on now has its own phase
labels independently confirmed, not just trusted. The deployment story
goes one step further than clustering alone: a handful of known customers
is enough to name the clusters, and a conformal confidence set on top of
that naming tells a utility exactly which assignments to trust and which
to send a field crew to check first, a statistically grounded answer none
of this chapter's own references provide. Part 4's own foundation is now complete: a real, solvable network, a
real time-series simulation of how it behaves under DER growth, and now
real, checked phase labels rather than assumed ones. Part 5 groups
customers and feeders by how they actually behave rather than how they
are labeled, reusing the same `ark.plot.clustering` tools this notebook
introduced.

## References